In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

In [9]:
import os
import re
import pandas as pd

root = r"C:\Users\cronk\.vscode\Sticky_Dumbbell_MS_Project\RDP_SF004"   # parent of the five sub‑folders
folders = sorted(
    os.path.join(root, d) for d in os.listdir(root)
    if os.path.isdir(os.path.join(root, d))
)

df_by_folder = {}        # will hold one DataFrame per folder

# regex to grab the DP number (e.g. 10DP, 15DP) from the filename
dp_re = re.compile(r"_(\d+DP)_")

for folder in folders:
    rows   = []          # list of one‑row DataFrames
    names  = []          # corresponding file names for the index
    for fname in sorted(os.listdir(folder)):
        # only read the "_parameters.csv" files
        if not fname.lower().endswith("_goodness.csv"):
            continue
        path = os.path.join(folder, fname)
        df = pd.read_csv(path)          # each file is assumed to contain exactly one row
        if len(df) != 1:
            raise ValueError(f"{path!r} has {len(df)} rows; expected 1")
        # --- extract DP and add as column ---
        m = dp_re.search(fname)
        if m:
            df['DP'] = m.group(1)
        else:
            df['DP'] = pd.NA
        # --- option A: add filename as a column ---
        df['set'] = os.path.splitext(fname)[0]
        rows.append(df)
        # --- option B: use names list for a MultiIndex below ---

        names.append(os.path.splitext(fname)[0])

    if not rows:
        continue

    # vertical concatenation: 5 files → 5 rows
    combined = pd.concat(rows, ignore_index=True)

    # if you prefer the filename as the row index instead of a column:
    # combined = pd.concat(rows, keys=names, names=['set'])

    df_by_folder[os.path.basename(folder)] = combined

# inspect one folder
print(df_by_folder['RDP_SF004_05A_data'])

      R2_Gp    R2_Gpp  R2_total  R2_log_Gp  R2_log_Gpp  R2_log_total  \
0  0.989260  0.639860  0.983422   0.987603    0.734991      0.968223   
1  0.989554  0.683960  0.983544   0.988471    0.789040      0.971536   
2  0.989669  0.685792  0.983671   0.988544    0.791646      0.971894   
3  0.989938  0.663393  0.983537   0.987765    0.792274      0.971647   
4  0.989227  0.686692  0.983054   0.988670    0.791740      0.970970   

      RMSE_Gp    RMSE_Gpp  Reduced_chi2         AIC    DP  \
0  390.913690  408.560395      0.025226  247.642019  10DP   
1  383.360567  400.325057      0.022835  366.265701  15DP   
2  381.207280  399.179668      0.021692  485.352626  20DP   
3  374.615315  405.484558      0.021098  604.705709  25DP   
4  397.731786  414.843798      0.023276  752.903147  31DP   

                           set  
0  fit_SF004_05A_10DP_goodness  
1  fit_SF004_05A_15DP_goodness  
2  fit_SF004_05A_20DP_goodness  
3  fit_SF004_05A_25DP_goodness  
4  fit_SF004_05A_31DP_goodness  


In [10]:
df_by_folder['RDP_SF004_15A_data']

,R2_Gp,R2_Gpp,R2_total,R2_log_Gp,R2_log_Gpp,R2_log_total,RMSE_Gp,RMSE_Gpp,Reduced_chi2,AIC,DP,set
0,0.991436,0.813526,0.985918,0.990790,0.902719,0.981548,366.815356,381.983232,0.021343,245.021119,10DP,fit_SF004_15A.bbx_10DP_goodness
1,0.991447,0.832371,0.985779,0.990977,0.919722,0.982905,362.803684,376.795091,0.019902,362.788374,15DP,fit_SF004_15A.bbx_15DP_goodness
2,0.991482,0.832350,0.985808,0.991084,0.920181,0.983061,361.899340,376.587551,0.018958,480.932314,20DP,fit_SF004_15A.bbx_20DP_goodness
3,0.991779,0.824428,0.985878,0.990635,0.921848,0.982961,354.472942,379.388476,0.018270,598.574904,25DP,fit_SF004_15A.bbx_25DP_goodness
4,0.991068,0.826760,0.985197,0.991094,0.917178,0.982439,378.321053,393.097691,0.020398,746.453181,31DP,fit_SF004_15A_31DP_goodness


In [ ]:
import matplotlib.pyplot as plt

# 5 data‑frames – replace these names with whatever you actually have in your
# namespace, or build the list dynamically.
dfs   = [df_10DP, df_15DP, df_20DP, df_25DP, df_original]
# optional: a list of labels for the legend, one per frame
labels = ['10DP', '15DP', '20DP', '25DP', 'original']
data_points = [10, 15, 20, 25, 31]
def plot_vs_dp(dfs, col, title, ylabel):
    i=0
    plt.figure()
    for df, lab in zip(dfs, labels):
        plt.plot(data_points[i],            # number of data points
                 df[col],             # quantity to plot
                 marker='o', linestyle='-',
                 label=lab)
        i += 1
    plt.xlabel('Number of data points (DP)')
    plt.ylabel(ylabel)
    plt.title(title)
    plt.legend()
    plt.grid(True)
    plt.tight_layout()

# three graphs
plot_vs_dp(dfs, 'R2_total',       'R²_total vs DP for SF004_05A',       'R²_total')
plt.savefig('SF004_05A_R2_total_vs_DP.png')   

plot_vs_dp(dfs, 'R2_log_total',   'R²_log_total vs DP for SF004_05A',   'R²_log_total')
plt.savefig('SF004_05A_R2_log_total_vs_DP.png')

plot_vs_dp(dfs, 'Reduced_chi2',   'Reduced χ² vs DP for SF004_05A',      'Reduced χ²')
plt.savefig('SF004_05A_Reduced_chi2_vs_DP.png')

plt.show()